In [0]:
--** useful in joins
SELECT NULL = NULL; --returns null
SELECT NULL<=> NULL; --returns true

In [0]:
%python
# PySpark equivalent of <=> is eqNullSafe()
from pyspark.sql.functions import col, lit

# Create sample DataFrame with NULLs
data = [
    (1, "Alice", "NY"),
    (2, "Bob", None),
    (3, "Charlie", None),
    (4, "Diana", "CA")
]
df = spark.createDataFrame(data, ["id", "name", "city"])

print("Original DataFrame:")
df.show()

# Regular equality (NULL == NULL returns None, filters out the row)
print("\nRegular equality (city == None) - excludes NULL matches:")
df.filter(col("city") == None).show()

# NULL-safe equality (NULL <=> NULL returns True, includes the row)
print("\nNULL-safe equality (city.eqNullSafe(None)) - includes NULL matches:")
df.filter(col("city").eqNullSafe(None)).show()

In [0]:
%python
# Practical use case: NULL-safe joins
# Useful when you want to match NULL values in join keys

from pyspark.sql.functions import col

# Create two DataFrames with NULL keys
df1 = spark.createDataFrame([
    (1, "Alice", "NY"),
    (2, "Bob", None),
    (3, "Charlie", "CA")
], ["id", "name", "state"])

df2 = spark.createDataFrame([
    ("NY", "New York"),
    (None, "Unknown"),
    ("CA", "California")
], ["state_code", "state_name"])

print("DataFrame 1:")
df1.show()

print("DataFrame 2:")
df2.show()

# Regular join - excludes NULL matches (Bob won't match)
print("\nRegular join (state == state_code) - Bob excluded:")
df1.join(df2, df1.state == df2.state_code, "left").show()

# NULL-safe join - includes NULL matches (Bob will match Unknown)
print("\nNULL-safe join (state <=> state_code) - Bob included:")
df1.join(df2, df1.state.eqNullSafe(df2.state_code), "left").show()

In [0]:
-- SQL equivalent: JOIN with NULL-safe equality

-- Create sample tables
CREATE OR REPLACE TEMP VIEW customers AS
SELECT 1 as id, 'Alice' as name, 'NY' as state
UNION ALL
SELECT 2 as id, 'Bob' as name, NULL as state
UNION ALL
SELECT 3 as id, 'Charlie' as name, 'CA' as state;

CREATE OR REPLACE TEMP VIEW states AS
SELECT 'NY' as code, 'New York' as full_name
UNION ALL
SELECT NULL as code, 'Unknown' as full_name
UNION ALL
SELECT 'CA' as code, 'California' as full_name;

-- Regular join: Bob excluded (NULL != NULL)
SELECT c.id, c.name, c.state, s.full_name
FROM customers c
LEFT JOIN states s ON c.state = s.code;

-- NULL-safe join: Bob included (NULL <=> NULL)
SELECT c.id, c.name, c.state, s.full_name
FROM customers c
LEFT JOIN states s ON c.state <=> s.code;